In [1]:
# Autoload modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import sys
sys.path.append('../')
from fcc import MiyazawaJerniganInteraction, Peptide, ProteinFoldingProblem, PenaltyParameters, ProteinSolver
import fcc
from qiskit.circuit.library import RealAmplitudes, EfficientSU2
from qiskit_aer.primitives import SamplerV2 as Sampler
import matplotlib.pyplot as plt
import numpy as np
import ray
import json

In [5]:
num_workers = 12
ray.init(
    num_cpus=num_workers,
    ignore_reinit_error=True,
    log_to_driver=False,
    runtime_env={
        "py_modules": [fcc],
    },
)

2024-11-04 21:17:55,766	INFO worker.py:1807 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2024-11-04 21:17:55,776	INFO packaging.py:600 -- Creating a file package for local module '/Users/lir9/protein-folding-qc/notebooks/../fcc'.
2024-11-04 21:17:55,782	INFO packaging.py:392 -- Pushing file package 'gcs://_ray_pkg_93ef65f8aef1d67d.zip' (0.19MiB) to Ray cluster...
2024-11-04 21:17:55,783	INFO packaging.py:405 -- Successfully pushed file package 'gcs://_ray_pkg_93ef65f8aef1d67d.zip'.


Python version:,3.12.2
Ray version:,2.38.0
Dashboard:,http://127.0.0.1:8265


In [ ]:
# ray.shutdown()

In [6]:
def build_pf(main_seq: str, energy_matrix_file: str = "mj_matrix"):
    """Builds the protein folding problem for the given sequence."""

    mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file)
    # print(mj_interaction.calculate_energy_matrix(main_seq))

    penalty_back = 50
    penalty_redun = 50
    penalty_olap = 50

    penalty_terms = PenaltyParameters(penalty_back, penalty_redun, penalty_olap)

    peptide = Peptide(main_seq)

    protein_folding_problem = ProteinFoldingProblem(
        peptide, mj_interaction, penalty_terms
    )

    return protein_folding_problem

In [7]:
# main_seq = "LHPGAGK" # Zika
main_seq = "LFLF" # 6MVZ ("LFLF")
protein_folding_problem = build_pf(main_seq)

qubit_op = protein_folding_problem.qubit_op()
print("Number of qubits: ", qubit_op.num_qubits)
print("Number of terms: ", len(qubit_op))
print(qubit_op)

Highest degree of the polynomial for 0 and 3: 6
Number of qubits:  9
Number of terms:  131
SparsePauliOp(['IIIIIIIII', 'IIIZZIIII', 'IIIZIZZII', 'IIIIZZZII', 'IIIIIIZZI', 'IIIZZIZZI', 'IIIZIZIZI', 'IIIIZZIZI', 'IIIIIZIIZ', 'IIIZZZIIZ', 'IIIZIIZIZ', 'IIIIZIZIZ', 'IIIIIZZZZ', 'IIIZZZZZZ', 'IIIZIIIZZ', 'IIIIZIIZZ', 'IIIIIZZII', 'IIIZZZZII', 'IIIIIIIZI', 'IIIIIZIII', 'IIIZIZIII', 'IIIIZZIII', 'IIIZZZIII', 'IIIIIIZII', 'IIIZIIZII', 'IIIIZIZII', 'IIIZZIZII', 'IIIIIZIZI', 'IIIZZZIZI', 'IIIZIIZZI', 'IIIZIIIII', 'IIIIZIIII', 'IIIIIIIZZ', 'IIIIIZIZZ', 'IIIZZZIZZ', 'IIIIIIZZZ', 'IIIZZIZZZ', 'IIIZIZZZZ', 'IIIIZZZZZ', 'IIIIZZIIZ', 'IIIIZIZZI', 'IIIZIIIZI', 'IIIIZIIZI', 'IIIZZIIZI', 'IIIIIZZZI', 'IIIZIZZZI', 'IIIIZZZZI', 'IIIZZZZZI', 'IIIIIIIIZ', 'IIIZIIIIZ', 'IIIIZIIIZ', 'IIIIIIZIZ', 'IIIZZIZIZ', 'IIIZIZZIZ', 'IIIIZZZIZ', 'IIIIZZIZZ', 'IIIIZIZZZ', 'IIIZIZIZZ', 'IIIZZIIZZ', 'IIIZIIZZZ', 'IIIZIZIIZ', 'IIIZZIIIZ', 'IIIZZZZIZ', 'IIIIIZZIZ', 'IIZIIIIII', 'IIZIIIIZI', 'IIZIIIIZZ', 'IZIIIIIII', 'IZIIIIIZI

Below we incorporate the sampling VQE approach, where we store all the configurations sampled during the VQE optimization. In the end, we choose the top 100 configurations as the final result and save them in a file.

In [9]:
shots = 10_000
ansatz = RealAmplitudes(qubit_op.num_qubits, reps=1).decompose()
sampler = Sampler(default_shots=shots)

optimizer = "COBYLA"
maxiter = 100
num_batches = num_workers

protein_solver = ProteinSolver(ansatz=ansatz, hamiltonian=qubit_op, sampler=sampler)
opt_results = protein_solver.train(optimizer=optimizer, maxiter=maxiter, num_batches=num_batches, verbose=True)

 >> Sampler job took 0.02 seconds
 >> Processing 229 unique bitstrings took 0.01 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 68 unique bitstrings took 0.01 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 24 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 24 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 24 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 9 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 33 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 59 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 4 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 0 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 >> Processing 1 unique bitstrings took 0.00 seconds
 >> Sampler job took 0.02 seconds
 

In [10]:
opt_results

{'final_cost': -16.34328671328653,
 'opt_params': array([11.1252752 ,  6.24828134, 12.13975753,  5.71329393,  8.88540797,
         9.51319399,  6.12852358,  9.03331526,  2.93202916,  7.96640568,
         3.10434341, 10.70089945,  5.49856685, 13.39922733, 12.38047039,
        12.6534286 ,  9.44848094,  9.1745399 ]),
 'cost_trajectory': [-8.652139160838992,
  -13.932704895104752,
  -10.372546853146732,
  -14.511246153845983,
  -13.518137062936928,
  -12.320111188811042,
  -13.441344755244645,
  -14.31127832167827,
  -13.914715384615256,
  -15.742439860139706,
  -7.740774825174813,
  -7.773998601398593,
  -10.445290209790025,
  -15.675104895104742,
  -16.005724475524307,
  -13.637288811188814,
  -16.34328671328653,
  -16.333790909090744,
  -16.34328671328653,
  -16.34328671328653,
  -16.34328671328653,
  -16.233146853146675,
  -13.637788811188637,
  -16.34328671328653,
  -16.34328671328653,
  -16.34328671328653,
  -16.34328671328653,
  -16.34328671328653,
  -16.34328671328653,
  -16.34328